<a href="https://colab.research.google.com/github/Neeti1987/Pathway-GEM-scripts/blob/main/Tryptophan_pathway_specific_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === STEP 1: Install dependencies ===
!pip install cobra pandas

import cobra
from cobra import Model, Reaction, Metabolite
import pandas as pd
from google.colab import files

# === STEP 2: Upload KO annotation file ===
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === STEP 3: Parse KO annotations genome-wise ===
genome_kos = {}
with open(filename, "r") as f:
    for line in f:
        if not line.strip(): continue
        parts = line.strip().split("\t")
        if len(parts) < 2: continue
        genome_id, ko = parts[0], parts[1]
        genome_prefix = genome_id.split("_")[0]
        genome_kos.setdefault(genome_prefix, set()).add(ko)

# === STEP 4: Apply KEGG logic definition for M00023 ===
def check_tryptophan_module(ko_list):
    status = {}

    # Block 1: Anthranilate synthase
    block1 = (
        (("K01657" in ko_list and "K01658" in ko_list) and "K00766" in ko_list) or
        ("K13503" in ko_list and "K00766" in ko_list) or
        ("K13501" in ko_list and "K00766" in ko_list) or
        ("K01656" in ko_list and "K00766" in ko_list) or
        ("K13497" in ko_list)
    )
    status["Anthranilate_synthase"] = block1

    # Block 2: Phosphoribosyltransferase + isomerase
    block2 = (
        ("K01817" in ko_list or "K24017" in ko_list) or
        ("K01656" in ko_list and "K01609" in ko_list) or
        ("K13498" in ko_list) or
        ("K13501" in ko_list)
    )
    status["Phosphoribosyltransferase"] = block2

    # Block 3: Tryptophan synthase
    block3 = (
        ("K01695" in ko_list and ("K01696" in ko_list or "K06001" in ko_list)) or
        ("K01694" in ko_list)
    )
    status["Tryptophan_synthase"] = block3

    complete = all(status.values())
    return complete, status

# === STEP 5: Build KEGG-accurate tryptophan pathway model ===
def build_tryptophan_model():
    model = Model("Tryptophan_pathway")

    # Metabolites (C numbers)
    chorismate = Metabolite("C00251_chorismate_c", compartment="c")
    nh3 = Metabolite("C00014_nh3_c", compartment="c")
    gln = Metabolite("C00064_gln_c", compartment="c")
    anthranilate = Metabolite("C00108_anthranilate_c", compartment="c")
    pyr = Metabolite("C00022_pyr_c", compartment="c")
    h2o = Metabolite("C00001_h2o_c", compartment="c")
    glu = Metabolite("C00025_glu_c", compartment="c")
    prpp = Metabolite("C00119_prpp_c", compartment="c")
    ppi = Metabolite("C00013_ppi_c", compartment="c")
    pra = Metabolite("C04302_pra_c", compartment="c")
    ribulose = Metabolite("C01302_ribulose_c", compartment="c")
    igp = Metabolite("C03506_igp_c", compartment="c")
    co2 = Metabolite("C00011_co2_c", compartment="c")
    ser = Metabolite("C00065_ser_c", compartment="c")
    trp = Metabolite("C00078_trp_c", compartment="c")
    g3p = Metabolite("C00118_g3p_c", compartment="c")

    # Reactions (R numbers)
    r00985 = Reaction("R00985_ammonia_variant")
    r00985.add_metabolites({chorismate:-1, nh3:-1, anthranilate:1, pyr:1, h2o:1})

    r00986 = Reaction("R00986_glutamine_variant")
    r00986.add_metabolites({chorismate:-1, gln:-1, anthranilate:1, pyr:1, glu:1})

    r01073 = Reaction("R01073_phosphoribosyltransferase")
    r01073.add_metabolites({anthranilate:-1, prpp:-1, pra:1, ppi:1})

    r03509 = Reaction("R03509_isomerase")
    r03509.add_metabolites({pra:-1, ribulose:1})

    r03508 = Reaction("R03508_igp_synthase")
    r03508.add_metabolites({ribulose:-1, igp:1, co2:1, h2o:1})

    r02722 = Reaction("R02722_tryptophan_synthase")
    r02722.add_metabolites({ser:-1, igp:-1, trp:1, g3p:1, h2o:1})

    model.add_reactions([r00985, r00986, r01073, r03509, r03508, r02722])

    # Exchanges for substrates/products
    for met in [chorismate, nh3, gln, prpp, ser, trp]:
        ex = Reaction(f"EX_{met.id}")
        ex.add_metabolites({met:1})
        ex.lower_bound = -1000
        ex.upper_bound = 1000
        model.add_reactions([ex])

    # Demand reaction for tryptophan
    dm_trp = Reaction("DM_C00078_trp_c")
    dm_trp.add_metabolites({trp:-1})
    dm_trp.lower_bound = 0
    dm_trp.upper_bound = 1000
    model.add_reactions([dm_trp])
    model.objective = dm_trp

    return model

# === STEP 6: Run genome-wise analysis ===
summary = []
for genome, kos in genome_kos.items():
    complete, status = check_tryptophan_module(kos)
    flux_value = None
    if complete:
        model = build_tryptophan_model()
        sol = model.optimize()
        flux_value = sol.objective_value
    summary.append({
        "Genome": genome,
        "Anthranilate_synthase": status["Anthranilate_synthase"],
        "Phosphoribosyltransferase": status["Phosphoribosyltransferase"],
        "Tryptophan_synthase": status["Tryptophan_synthase"],
        "Module_complete": complete,
        "Flux_to_tryptophan": flux_value
    })

# === STEP 7: Output results ===
df = pd.DataFrame(summary)
print(df)
df.to_csv("tryptophan_flux_genomewise.tsv", sep="\t", index=False)

# === STEP 8: Download TSV ===
files.download("tryptophan_flux_genomewise.tsv")


Saving FINAL BLAST.tsv to FINAL BLAST (1).tsv
    Genome  Anthranilate_synthase  Phosphoribosyltransferase  \
0    ADCAI                  False                       True   
1    AFCAI                   True                       True   
2      BTB                   True                       True   
3      BTQ                   True                       True   
4   BTQVLC                   True                       True   
5     BTZ1                   True                       True   
6    China                   True                       True   
7   China1                   True                       True   
8    MEAM1                   True                       True   
9     PeMo                   True                       True   
10    SiSi                   True                      False   
11      TV                   True                       True   

    Tryptophan_synthase  Module_complete  Flux_to_tryptophan  
0                  True            False                 N

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from cobra import Model, Reaction, Metabolite

# --- Threonine (M00018) ---
def build_thr_model():
    model = Model("Threonine_pathway")

    # Metabolites
    asp = Metabolite("C00049_asp_c", compartment="c")
    atp = Metabolite("C00002_atp_c", compartment="c")
    adp = Metabolite("C00008_adp_c", compartment="c")
    asa_phosphate = Metabolite("C03082_asa_phosphate_c", compartment="c")
    asa_semialdehyde = Metabolite("C00441_asa_semialdehyde_c", compartment="c")
    nadp = Metabolite("C00006_nadp_c", compartment="c")
    nadph = Metabolite("C00005_nadph_c", compartment="c")
    nad = Metabolite("C00003_nad_c", compartment="c")
    nadh = Metabolite("C00004_nadh_c", compartment="c")
    h = Metabolite("C00080_h_c", compartment="c")
    h2o = Metabolite("C00001_h2o_c", compartment="c")
    pi = Metabolite("C00009_pi_c", compartment="c")
    hser = Metabolite("C00263_hser_c", compartment="c")
    phser = Metabolite("C01102_phser_c", compartment="c")
    thr = Metabolite("C00188_thr_c", compartment="c")

    # Reactions
    r00480 = Reaction("R00480_Aspartokinase")
    r00480.add_metabolites({asp:-1, atp:-1, adp:1, asa_phosphate:1})

    r02291 = Reaction("R02291_Aspartate_semialdehyde_dehydrogenase")
    r02291.add_metabolites({asa_semialdehyde:-1, pi:-1, nadp:-1,
                            asa_phosphate:1, nadph:1, h:1})

    r01773 = Reaction("R01773_Homoserine_dehydrogenase_NAD")
    r01773.add_metabolites({hser:-1, nad:-1, asa_semialdehyde:1, nadh:1, h:1})

    r01775 = Reaction("R01775_Homoserine_dehydrogenase_NADP")
    r01775.add_metabolites({hser:-1, nadp:-1, asa_semialdehyde:1, nadph:1, h:1})

    r01771 = Reaction("R01771_Homoserine_kinase")
    r01771.add_metabolites({hser:-1, atp:-1, adp:1, phser:1})

    r01466 = Reaction("R01466_Threonine_synthase")
    r01466.add_metabolites({phser:-1, h2o:-1, thr:1, pi:1})

    model.add_reactions([r00480, r02291, r01773, r01775, r01771, r01466])

    # Demand reaction
    dm_thr = Reaction("DM_C00188_thr_c")
    dm_thr.add_metabolites({thr:-1})
    dm_thr.lower_bound = 0; dm_thr.upper_bound = 1000
    model.add_reactions([dm_thr])
    model.objective = dm_thr
    return model


# --- Tryptophan (M00023) ---
def build_trp_model():
    model = Model("Tryptophan_pathway")

    # Metabolites
    chorismate = Metabolite("C00251_chorismate_c", compartment="c")
    nh3 = Metabolite("C00014_nh3_c", compartment="c")
    gln = Metabolite("C00064_gln_c", compartment="c")
    anthranilate = Metabolite("C00108_anthranilate_c", compartment="c")
    pyr = Metabolite("C00022_pyr_c", compartment="c")
    h2o = Metabolite("C00001_h2o_c", compartment="c")
    glu = Metabolite("C00025_glu_c", compartment="c")
    prpp = Metabolite("C00119_prpp_c", compartment="c")
    ppi = Metabolite("C00013_ppi_c", compartment="c")
    pra = Metabolite("C04302_pra_c", compartment="c")
    ribulose = Metabolite("C01302_ribulose_c", compartment="c")
    igp = Metabolite("C03506_igp_c", compartment="c")
    co2 = Metabolite("C00011_co2_c", compartment="c")
    ser = Metabolite("C00065_ser_c", compartment="c")
    trp = Metabolite("C00078_trp_c", compartment="c")
    g3p = Metabolite("C00118_g3p_c", compartment="c")

    # Reactions
    r00985 = Reaction("R00985_ammonia_variant")
    r00985.add_metabolites({chorismate:-1, nh3:-1, anthranilate:1, pyr:1, h2o:1})

    r00986 = Reaction("R00986_glutamine_variant")
    r00986.add_metabolites({chorismate:-1, gln:-1, anthranilate:1, pyr:1, glu:1})

    r01073 = Reaction("R01073_phosphoribosyltransferase")
    r01073.add_metabolites({anthranilate:-1, prpp:-1, pra:1, ppi:1})

    r03509 = Reaction("R03509_isomerase")
    r03509.add_metabolites({pra:-1, ribulose:1})

    r03508 = Reaction("R03508_igp_synthase")
    r03508.add_metabolites({ribulose:-1, igp:1, co2:1, h2o:1})

    r02722 = Reaction("R02722_tryptophan_synthase")
    r02722.add_metabolites({ser:-1, igp:-1, trp:1, g3p:1, h2o:1})

    model.add_reactions([r00985, r00986, r01073, r03509, r03508, r02722])

    # Demand reaction
    dm_trp = Reaction("DM_C00078_trp_c")
    dm_trp.add_metabolites({trp:-1})
    dm_trp.lower_bound = 0; dm_trp.upper_bound = 1000
    model.add_reactions([dm_trp])
    model.objective = dm_trp
    return model


# --- Methionine (M00017) ---
def build_met_model():
    model = Model("Methionine_pathway")

    # Metabolites
    hser = Metabolite("C00263_hser_c", compartment="c")
    asa_semialdehyde = Metabolite("C00441_asa_semialdehyde_c", compartment="c")
    osucc_hser = Metabolite("C01118_osucc_hser_c", compartment="c")
    cystathionine = Metabolite("C02291_cystathionine_c", compartment="c")
    homocys = Metabolite("C00155_homocys_c", compartment="c")
    folate = Metabolite("C00440_folate_c", compartment="c")
    folate_tri = Metabolite("C04489_folate_tri_c", compartment="c")
    met = Metabolite("C00073_met_c", compartment="c")

    # Reactions
    r01773 = Reaction("R01773_Homoserine_dehydrogenase_NAD")
    r01773.add_metabolites({hser:-1, asa_semialdehyde:1})

    r01777 = Reaction("R01777_Homoserine_succinyltransferase")
    r01777.add_metabolites({hser:-1, osucc_hser:1})

    r03260 = Reaction("R03260_Cystathionine_gamma_synthase")
    r03260.add_metabolites({osucc_hser:-1, cystathionine:1})

    r01286 = Reaction("R01286_Cystathionine_lyase")
    r01286.add_metabolites({cystathionine:-1, homocys:1})

    r00946 = Reaction("R00946_Methionine_synthase")
    r00946.add_metabolites({homocys:-1, folate:-1, met:1})

    r04405 = Reaction("R04405_Methionine_synthase_alt")
    r04405.add_metabolites({homocys:-1, folate_tri:-1, met:1})

        # Add reactions
    model.add_reactions([r01773, r01777, r03260, r01286, r00946, r04405])

    # Demand reaction for methionine
    dm_met = Reaction("DM_C00073_met_c")
    dm_met.add_metabolites({met:-1})
    dm_met.lower_bound = 0
    dm_met.upper_bound = 1000
    model.add_reactions([dm_met])
    model.objective = dm_met

    return model




In [ ]:
from cobra import Model, Reaction, Metabolite

def build_leu_model():
    model = Model("Leucine_pathway")

    # Metabolites
    oxoisovalerate = Metabolite("C00141_oxoisovalerate_c", compartment="c")
    acetylcoa = Metabolite("C00024_acetylcoa_c", compartment="c")
    isopropylmalate = Metabolite("C02504_isopropylmalate_c", compartment="c")
    isopropylmalate3 = Metabolite("C04411_isopropylmalate3_c", compartment="c")
    isopropylmaleate = Metabolite("C02631_isopropylmaleate_c", compartment="c")
    coa = Metabolite("C00010_coa_c", compartment="c")
    h2o = Metabolite("C00001_h2o_c", compartment="c")
    nad = Metabolite("C00003_nad_c", compartment="c")
    nadh = Metabolite("C00004_nadh_c", compartment="c")
    h = Metabolite("C00080_h_c", compartment="c")
    oxosuccinate = Metabolite("C04236_oxosuccinate_c", compartment="c")
    co2 = Metabolite("C00011_co2_c", compartment="c")
    oxoisocaproate = Metabolite("C00233_oxoisocaproate_c", compartment="c")

    # Reactions
    r01213 = Reaction("R01213_Isopropylmalate_synthase")
    r01213.add_metabolites({isopropylmalate:-1, coa:-1, acetylcoa:1, oxoisovalerate:1, h2o:1})

    r03968 = Reaction("R03968_Isopropylmalate_hydrolyase")
    r03968.add_metabolites({isopropylmalate:-1, isopropylmaleate:1, h2o:1})

    r04001 = Reaction("R04001_3Isopropylmalate_hydrolyase")
    r04001.add_metabolites({isopropylmalate3:-1, isopropylmaleate:1, h2o:1})

    r04426 = Reaction("R04426_Isopropylmalate_dehydrogenase")
    r04426.add_metabolites({isopropylmalate3:-1, nad:-1, oxosuccinate:1, nadh:1, h:1})

    r01652 = Reaction("R01652_Oxoisocaproate_carboxylation")
    r01652.add_metabolites({oxoisocaproate:-1, co2:-1, oxosuccinate:1})

    model.add_reactions([r01213, r03968, r04001, r04426, r01652])

    # Demand reaction
    dm_leu = Reaction("DM_C00233_oxoisocaproate_c")
    dm_leu.add_metabolites({oxoisocaproate:-1})
    dm_leu.lower_bound = 0
    dm_leu.upper_bound = 1000
    model.add_reactions([dm_leu])
    model.objective = dm_leu

    return model


In [ ]:
from cobra import Model, Reaction, Metabolite

def build_lys_model():
    model = Model("Lysine_pathway")

    # Metabolites
    asp = Metabolite("C00049_asp_c", compartment="c")
    atp = Metabolite("C00002_atp_c", compartment="c")
    adp = Metabolite("C00008_adp_c", compartment="c")
    phospho_asp = Metabolite("C03082_phospho_asp_c", compartment="c")
    semialdehyde = Metabolite("C00441_semialdehyde_c", compartment="c")
    pi = Metabolite("C00009_pi_c", compartment="c")
    nadp = Metabolite("C00006_nadp_c", compartment="c")
    nadph = Metabolite("C00005_nadph_c", compartment="c")
    nad = Metabolite("C00003_nad_c", compartment="c")
    nadh = Metabolite("C00004_nadh_c", compartment="c")
    h = Metabolite("C00080_h_c", compartment="c")
    pyr = Metabolite("C00022_pyr_c", compartment="c")
    htpa = Metabolite("C20258_htpa_c", compartment="c")
    tetrahydro_dap = Metabolite("C03972_tetrahydro_dap_c", compartment="c")
    succinylcoa = Metabolite("C00091_succinylcoa_c", compartment="c")
    coa = Metabolite("C00010_coa_c", compartment="c")
    succinyl_dap = Metabolite("C04462_succinyl_dap_c", compartment="c")
    succinyl_diamino = Metabolite("C04421_succinyl_diamino_c", compartment="c")
    oxoglutarate = Metabolite("C00026_2oxoglutarate_c", compartment="c")
    glu = Metabolite("C00025_glu_c", compartment="c")
    succinate = Metabolite("C00042_succinate_c", compartment="c")
    ll_dap = Metabolite("C00666_ll_dap_c", compartment="c")
    meso_dap = Metabolite("C00680_meso_dap_c", compartment="c")
    lys = Metabolite("C00047_lys_c", compartment="c")
    co2 = Metabolite("C00011_co2_c", compartment="c")
    h2o = Metabolite("C00001_h2o_c", compartment="c")

    # Reactions
    r00480 = Reaction("R00480_Aspartokinase")
    r00480.add_metabolites({asp:-1, atp:-1, adp:1, phospho_asp:1})

    r02291 = Reaction("R02291_Aspartate_semialdehyde_dehydrogenase")
    r02291.add_metabolites({semialdehyde:-1, pi:-1, nadp:-1, phospho_asp:1, nadph:1, h:1})

    r10147 = Reaction("R10147_HTPA_synthase")
    r10147.add_metabolites({semialdehyde:-1, pyr:-1, htpa:1, h2o:1})

    r04198 = Reaction("R04198_HTPA_reductase_NAD")
    r04198.add_metabolites({tetrahydro_dap:-1, nad:-1, h2o:-1, htpa:1, nadh:1, h:1})

    r04199 = Reaction("R04199_HTPA_reductase_NADP")
    r04199.add_metabolites({tetrahydro_dap:-1, nadp:-1, h2o:-1, htpa:1, nadph:1, h:1})

    r04365 = Reaction("R04365_Succinyltransferase")
    r04365.add_metabolites({succinylcoa:-1, tetrahydro_dap:-1, h2o:-1, coa:1, succinyl_dap:1})

    r04475 = Reaction("R04475_Aminotransferase")
    r04475.add_metabolites({succinyl_diamino:-1, oxoglutarate:-1, succinyl_dap:1, glu:1})

    r02734 = Reaction("R02734_Desuccinylase")
    r02734.add_metabolites({succinyl_diamino:-1, h2o:-1, succinate:1, ll_dap:1})

    r02735 = Reaction("R02735_Epimerase")
    r02735.add_metabolites({ll_dap:-1, meso_dap:1})

    r00451 = Reaction("R00451_DAP_decarboxylase")
    r00451.add_metabolites({meso_dap:-1, lys:1, co2:1})

    model.add_reactions([r00480, r02291, r10147, r04198, r04199, r04365, r04475, r02734, r02735, r00451])

    # Demand reaction
    dm_lys = Reaction("DM_C00047_lys_c")
    dm_lys.add_metabolites({lys:-1})
    dm_lys.lower_bound = 0
    dm_lys.upper_bound = 1000
    model.add_reactions([dm_lys])
    model.objective = dm_lys

    return model


In [ ]:
from cobra import Model, Reaction, Metabolite

def build_arg_model():
    model = Model("Arginine_pathway")

    # Metabolites
    carbamoyl_phosphate = Metabolite("C00169_carbamoyl_phosphate_c", compartment="c")
    ornithine = Metabolite("C00077_ornithine_c", compartment="c")
    pi = Metabolite("C00009_pi_c", compartment="c")
    citrulline = Metabolite("C00327_citrulline_c", compartment="c")
    asp = Metabolite("C00049_aspartate_c", compartment="c")
    atp = Metabolite("C00002_atp_c", compartment="c")
    amp = Metabolite("C00020_amp_c", compartment="c")
    ppi = Metabolite("C00013_ppi_c", compartment="c")
    arginosuccinate = Metabolite("C03406_argininosuccinate_c", compartment="c")
    arg = Metabolite("C00062_arginine_c", compartment="c")
    fumarate = Metabolite("C00122_fumarate_c", compartment="c")

    # Reactions
    r01398 = Reaction("R01398_Ornithine_carbamoyltransferase")
    r01398.add_metabolites({carbamoyl_phosphate:-1, ornithine:-1, pi:1, citrulline:1})

    r01954 = Reaction("R01954_Arginosuccinate_synthase")
    r01954.add_metabolites({atp:-1, citrulline:-1, asp:-1, amp:1, ppi:1, arginosuccinate:1})

    r01086 = Reaction("R01086_Arginosuccinate_lyase")
    r01086.add_metabolites({arginosuccinate:-1, arg:1, fumarate:1})

    model.add_reactions([r01398, r01954, r01086])

    # Demand reaction for arginine
    dm_arg = Reaction("DM_C00062_arginine_c")
    dm_arg.add_metabolites({arg:-1})
    dm_arg.lower_bound = 0
    dm_arg.upper_bound = 1000
    model.add_reactions([dm_arg])
    model.objective = dm_arg

    return model
